# Recompute the benchmark head-to-head on CPU

This notebook rebuilds the rows of the MetaTime head-to-head from the committed per-task records in `results/metatime/` and checks them against `results/metatime.json`. It needs no GPU, no model, no benchmark download and no credentials. Each system is scored against its own measured time. The records hold task ids and numbers, not the benchmark's task text (`docs/METATIME.md`).

In [ ]:
import csv, json, statistics, sys, tempfile
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'results/metatime.json').exists())
sys.path.insert(0, str(ROOT / 'scripts/metatime'))
import figure, metrics

RECORDS = ROOT / 'results/metatime'
result = json.loads((ROOT / 'results/metatime.json').read_text())
load = lambda name: [json.loads(line) for line in (RECORDS / name).read_text().splitlines() if line]
with (RECORDS / 'reference-published.csv').open(encoding='utf-8') as handle:
    published = list(csv.DictReader(handle))
runs = {}
for r in load('runs.jsonl'):
    runs.setdefault((r['executor'], r['task']), []).append(r['wall_s'])
groups = {}
for r in load('estimates.jsonl'):
    groups.setdefault(r['estimator'] + '/' + r['variant'], []).append(r)
# An adapter asked with no stated speed gives one set of answers, recorded once under
# `<name>@base` and also scored against the adapter's own timed runs as `<name>`.
for key in list(groups):
    name, _, variant = key.partition('/')
    if name.endswith('@base'):
        groups.setdefault(name.removesuffix('@base') + '/' + variant, groups[key])
print(sum(map(len, runs.values())), 'timed runs,', len(load('estimates.jsonl')), 'answers,',
      len(published), 'published rows')

## 1. Every row that was asked and timed

The two published rows are recomputed from the benchmark artifact's own per-task numbers. Ours are recomputed from our answers and the median of the timed runs they are scored against. Rows that replay an answer at another stated speed are computed targets, not runs, and are left to `scripts/metatime/score.py`.

In [ ]:
FIELDS = ('median_ratio', 'median_abs_log_error', 'pearson_log', 'within_2x', 'mae_s')

def differs(got, released):
    return any((got[f] is None) != (released[f] is None) or
               (got[f] is not None and got[f] != released[f] and abs(got[f] - released[f]) > 1e-9)
               for f in FIELDS)

table, bad = [], 0
for model, released in result['published'].items():
    mine = [r for r in published if r['model'] == model]
    got = metrics.metrics([float(r['estimate_before_seconds']) for r in mine],
                          [float(r['actual_seconds']) for r in mine])
    bad += differs(got, released)
    table.append({'system': model + ' (published)', 'scored against': 'its own runs', **{f: got[f] for f in FIELDS}})
for key, group in groups.items():
    system = result['systems'].get(key)
    if system is None or system['replayed_at_stated_speed']:
        continue
    actual = [statistics.median(runs[(system['scored_against'], r['task'])]) for r in group]
    got = metrics.metrics([r['stated_seconds'] for r in group], actual)
    bad += differs(got, system['stated'])
    table.append({'system': key, 'scored against': system['scored_against'] + ' runs', **{f: got[f] for f in FIELDS}})
print(len(table), 'rows recomputed;', bad, 'differ from results/metatime.json')
assert bad == 0
display(pd.DataFrame(table).round(3))

## 2. The declared comparison

The success criterion, declared before the first estimate and applied to the CuratorKIT → AlignTune adapter before its own runs (`docs/PREREGISTRATION.md`): the primary row (`g1-kit/card_description`) has a lower median absolute log error than each frontier model, and the paired 95% interval of the difference excludes zero. The pairing is by task; each system keeps its own measured times.

In [ ]:
PRIMARY = 'g1-kit/card_description'
primary = result['systems'][PRIMARY]
stated = {r['task']: r['stated_seconds'] for r in groups[PRIMARY]}
order = list(stated)  # the benchmark's task order, which fixes the bootstrap draws
rows = []
for model in result['published']:
    by_task = {r['task']: r for r in published if r['model'] == model}
    reference = [by_task[t] for t in order]
    got = metrics.paired_log_error(
        [stated[t] for t in order],
        [statistics.median(runs[(primary['scored_against'], t)]) for t in order],
        [float(r['estimate_before_seconds']) for r in reference],
        [float(r['actual_seconds']) for r in reference])
    released = primary['stated']['versus'][model]
    assert abs(got['difference'] - released['difference']) < 1e-9
    assert max(abs(a - b) for a, b in zip(got['ci95'], released['ci95'])) < 1e-9
    assert got['tasks_a_closer'] == released['tasks_a_closer']
    rows.append({'ours against': model, 'difference in median abs log error': got['difference'],
                 'CI95 low': got['ci95'][0], 'CI95 high': got['ci95'][1],
                 'tasks where ours is closer': f"{got['tasks_a_closer']} of {got['tasks']}"})
display(pd.DataFrame(rows).round(3))
print('criterion met:', all(row['CI95 high'] < 0 for row in rows))

## 3. The figure

Drawn from the same records by `scripts/metatime/figure.py`; the committed copy is `paper/figures/F9-metatime-head-to-head.png`.

In [ ]:
Image(filename=str(figure.head_to_head(ROOT, Path(tempfile.mkdtemp()))), width=1000)